## TRANSFORMATION BRONZE → SILVER

In [ ]:
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

#LORSQU'ON TRAVAILLE DEPUIS SA MACHINE LOCAL
MINIO_ENDPOINT = "http://192.168.1.230:30137"
MINIO_ACCESS_KEY = "datalab-team"
MINIO_SECRET_KEY = "minio-datalabteam123"
NESSIE_URI = "http://192.168.1.230:30604/api/v1"

#---------------------------------------------------------------------------------

#LORSQU'ON TRAVAILLE SUR JHUB
# MINIO_ENDPOINT = "http://minio.mon-namespace.svc.cluster.local:80"
# MINIO_ACCESS_KEY = "datalab-team"
# MINIO_SECRET_KEY = "minio-datalabteam123"
# NESSIE_URI = "http://nessie.trino.svc.cluster.local:19120/api/v1"

#---------------------------------------------------------------------------------

conf = (
    SparkConf()
    .setAppName("Bronze_to_Silver")
    .set("spark.driver.host", "127.0.0.1")
    .set("spark.driver.bindAddress", "127.0.0.1")
    .set("spark.driver.memory", "16g")
    .set("spark.jars.packages",
         "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1,"
         "org.apache.hadoop:hadoop-aws:3.3.4,"
         "org.projectnessie.nessie-integrations:nessie-spark-extensions-3.5_2.12:0.77.1")

    .set("spark.sql.extensions",
         "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,"
         "org.projectnessie.spark.extensions.NessieSparkSessionExtensions")

    # CONFIGURATION DU CATALOGUE NESSIE
    .set("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog")
    .set("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog")
    .set("spark.sql.catalog.nessie.uri", NESSIE_URI)
    .set("spark.sql.catalog.nessie.ref", "main")
    .set("spark.sql.catalog.nessie.warehouse", "s3a://bronze/")

    # CONFIGURATION MINIO (S3A)
    .set("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .set("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .set("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .set("spark.hadoop.fs.s3a.path.style.access", "true")
    .set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .set("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
)

In [ ]:
spark = SparkSession.builder.config(conf=conf).getOrCreate()

In [ ]:
# Lecture de la table Bronze
TABLE_BRONZE = "nessie.bronze.<<nom_de_la_table>>"

df = spark.table(TABLE_BRONZE)
print(f"Bronze : {df.count():,} lignes × {len(df.columns)} colonnes")
df.printSchema()

## ON EFFECTUE LES TRANSFORMATIONS SILVER ICI

Silver = données propres, typées, déduplicées, avec règles métier.  
On peut travailler en **DataFrame API** (ci-dessous) ou en **SQL pur** (cellule suivante).

In [ ]:
# --- OPTION A : DataFrame API ---

df_silver = (
    df
    # 1. Cast des types (Bronze = tout String)
    # .withColumn("montant_brut", F.col("montant_brut").cast("double"))
    # .withColumn("periode",      F.to_date("periode", "yyyy-MM"))

    # 2. Nettoyage
    # .withColumn("matricule", F.trim(F.upper(F.col("matricule"))))

    # 3. Filtrage lignes invalides
    # .filter(F.col("matricule").isNotNull())

    # 4. Colonnes calculées
    # .withColumn("annee", F.year("periode"))
    # .withColumn("mois",  F.month("periode"))

    # 5. Déduplication
    # .dropDuplicates(["matricule", "periode"])
)

In [ ]:
# --- OPTION B : SQL pur ---

# df.createOrReplaceTempView("bronze_source")
#
# df_silver = spark.sql("""
#     SELECT
#         TRIM(UPPER(matricule))           AS matricule,
#         CAST(montant_brut AS DOUBLE)     AS montant_brut,
#         CAST(montant_net  AS DOUBLE)     AS montant_net,
#         TO_DATE(periode, 'yyyy-MM')      AS periode,
#         YEAR(TO_DATE(periode, 'yyyy-MM')) AS annee
#     FROM bronze_source
#     WHERE matricule IS NOT NULL
#       AND CAST(montant_brut AS DOUBLE) > 0
# """)

In [ ]:
# Vérification avant écriture
print(f"Silver : {df_silver.count():,} lignes × {len(df_silver.columns)} colonnes")
df_silver.show(5, truncate=False)

In [ ]:
TABLE_SILVER = "nessie.silver.<<nom_de_la_table>>"

spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.silver")

df_silver.write \
    .format("iceberg") \
    .mode("overwrite") \
    .saveAsTable(TABLE_SILVER)